In [1]:
import sys
import sqlite3
import shutil
from pathlib import Path

print("===== ENVIRONMENT CHECK =====")

print("Python version:", sys.version)
print("Current folder:", Path.cwd())
print("SQLite version:", sqlite3.sqlite_version)
print("Git installed:", shutil.which("git") is not None)

try:
    import pandas as pd
    print("Pandas version:", pd.__version__)
    print("Pandas status: READY")
except ImportError:
    print("Pandas status: NOT INSTALLED")

print("\n===== CHECK DATA FOLDERS =====")

for folder_name in ["source-data", "data-hackathon", "data"]:
    folder = Path.cwd() / folder_name

    if folder.exists():
        print(f"\nFound folder: {folder_name}")

        for file in sorted(folder.iterdir()):
            if file.is_file():
                print("  ", file.name)
    else:
        print(f"{folder_name}: not found")

===== ENVIRONMENT CHECK =====
Python version: 3.14.7 (v3.14.7:823f0323ee6, Aug  5 2026, 07:07:01) [Clang 21.0.0 (clang-2100.1.1.101)]
Current folder: /
SQLite version: 3.50.4
Git installed: True
Pandas version: 3.1.0
Pandas status: READY

===== CHECK DATA FOLDERS =====
source-data: not found
data-hackathon: not found
data: not found


In [2]:
from pathlib import Path
import subprocess

# Mac ke Documents folder mein source repository rakhenge
source_dir = Path.home() / "Documents" / "data-hackathon"

# Documents folder ensure karo
source_dir.parent.mkdir(parents=True, exist_ok=True)

print("Dataset location:", source_dir)

if (source_dir / ".git").exists():
    print("\nRepository pehle se downloaded hai.")

elif source_dir.exists():
    print(
        "\nERROR: data-hackathon naam ka folder pehle se hai, "
        "lekin Git repository nahi hai."
    )

else:
    print("\nGitHub se repository download ho rahi hai...")

    result = subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/aiwithqasim/data-hackathon.git",
            str(source_dir)
        ],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("Download successful!")
        print(result.stdout)
    else:
        print("Download failed.")
        print(result.stderr)

# Download hui files ki list dikhao
if (source_dir / ".git").exists():
    print("\n===== REPOSITORY FILES =====")

    for path in sorted(source_dir.iterdir()):
        if path.is_file():
            print(path.name)

    print("\nCSV files:", len(list(source_dir.glob("*.csv"))))
    print("JSON files:", len(list(source_dir.glob("*.json"))))
    print(
        "Data dictionary exists:",
        (source_dir / "data_dictionary.md").exists()
    )

Dataset location: /Users/abdullahahmed/Documents/data-hackathon

GitHub se repository download ho rahi hai...
Download successful!


===== REPOSITORY FILES =====
applications.csv
biometric_punches.csv
data_dictionary.md
departments.csv
device_mapping.csv
employees.csv
employment_history.csv
holidays.csv
leave_requests.json
overtime_requests.csv
payroll_lines.csv
shift_roster.csv
shifts.json

CSV files: 10
JSON files: 2
Data dictionary exists: True


In [3]:
from pathlib import Path
import shutil

# Sir ki downloaded repository
source_dir = (
    Path.home()
    / "Documents"
    / "data-hackathon"
)

# Hamara apna project
project_dir = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

# Project ki folders
folders = [
    "data/raw",
    "docs",
    "ingestion",
    "transform",
    "warehouse/ddl",
    "analytics/results",
    "quality",
    "presentation"
]

# Folders create karo
for folder_name in folders:
    (project_dir / folder_name).mkdir(
        parents=True,
        exist_ok=True
    )

# Original data files ki copies banao
copied_files = []

for pattern in ["*.csv", "*.json"]:
    for source_file in source_dir.glob(pattern):
        destination = (
            project_dir / "data" / "raw"
            / source_file.name
        )

        if not destination.exists():
            shutil.copy2(source_file, destination)
            copied_files.append(source_file.name)

# Data dictionary ko docs folder mein rakho
dictionary_source = source_dir / "data_dictionary.md"
dictionary_destination = (
    project_dir / "docs" / "data_dictionary.md"
)

if not dictionary_destination.exists():
    shutil.copy2(
        dictionary_source,
        dictionary_destination
    )

print("PROJECT SETUP COMPLETE")
print("Project location:", project_dir)
print("New data files copied:", len(copied_files))

print("\n===== RAW DATA FILES =====")

raw_dir = project_dir / "data" / "raw"

for file in sorted(raw_dir.iterdir()):
    if file.is_file():
        print(file.name)

print("\nCSV files:", len(list(raw_dir.glob("*.csv"))))
print("JSON files:", len(list(raw_dir.glob("*.json"))))
print(
    "Data dictionary exists:",
    dictionary_destination.exists()
)

PROJECT SETUP COMPLETE
Project location: /Users/abdullahahmed/Documents/indus-weave-hr-pipeline
New data files copied: 12

===== RAW DATA FILES =====
applications.csv
biometric_punches.csv
departments.csv
device_mapping.csv
employees.csv
employment_history.csv
holidays.csv
leave_requests.json
overtime_requests.csv
payroll_lines.csv
shift_roster.csv
shifts.json

CSV files: 10
JSON files: 2
Data dictionary exists: True


In [4]:
from pathlib import Path
import json
import pandas as pd

# Hamare project aur raw data ki locations
PROJECT_DIR = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
DOCS_DIR = PROJECT_DIR / "docs"

DOCS_DIR.mkdir(parents=True, exist_ok=True)

# PDF ke mutabiq expected record counts
EXPECTED_COUNTS = {
    "employees.csv": 1316,
    "employment_history.csv": 3686,
    "departments.csv": 25,
    "shifts.json": 4,
    "shift_roster.csv": 36402,
    "device_mapping.csv": 1260,
    "biometric_punches.csv": 345280,
    "holidays.csv": 12,
    "leave_requests.json": 5372,
    "overtime_requests.csv": 22098,
    "payroll_lines.csv": 53617,
    "applications.csv": 3835,
}

audit_records = []

print("=" * 65)
print("PEOPLEOPS HACKATHON - SOURCE DATA AUDIT")
print("=" * 65)

# --------------------------------------------------
# PART 1: CSV FILES
# --------------------------------------------------

csv_files = sorted(RAW_DIR.glob("*.csv"))

print(f"\nCSV files found: {len(csv_files)}")

for file_path in csv_files:

    # Read data as text; original file ko change nahi karna
    df = pd.read_csv(
        file_path,
        dtype="string",
        keep_default_na=False
    )

    actual_rows = len(df)
    expected_rows = EXPECTED_COUNTS[file_path.name]

    # Empty ya whitespace-only cells
    blank_cells = int(
        df.apply(
            lambda column: column.str.strip().eq("").sum()
        ).sum()
    )

    # Completely identical rows; business-key duplicates nahi
    full_duplicate_rows = int(df.duplicated().sum())

    count_matches = actual_rows == expected_rows

    audit_records.append({
        "file_name": file_path.name,
        "format": "CSV",
        "actual_records": actual_rows,
        "expected_records": expected_rows,
        "count_matches": count_matches,
        "column_count": len(df.columns),
        "blank_cells": blank_cells,
        "fully_duplicate_rows": full_duplicate_rows,
        "json_keys": "",
        "json_list_members": "",
    })

    print(f"\nFile: {file_path.name}")
    print(f"Rows: {actual_rows:,}")
    print(f"Expected rows: {expected_rows:,}")
    print(f"Count matches: {count_matches}")
    print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
    print(f"Blank cells: {blank_cells:,}")
    print(f"Fully duplicated rows: {full_duplicate_rows:,}")

# --------------------------------------------------
# PART 2: JSON FILES
# --------------------------------------------------

json_files = sorted(RAW_DIR.glob("*.json"))

print(f"\n\nJSON files found: {len(json_files)}")

def inspect_json_records(obj):

    # JSON ki top level list ho
    if isinstance(obj, list):
        return len(obj), "top-level list"

    # JSON ki top level dictionary ho
    if isinstance(obj, dict):

        # Common names of record collections
        preferred_keys = [
            "leave_requests",
            "shifts",
            "requests",
            "data",
            "records",
        ]

        for key in preferred_keys:
            if isinstance(obj.get(key), list):
                return len(obj[key]), f"list inside '{key}'"

        # Agar dictionary mein sirf ek list ho
        list_members = [
            (key, value)
            for key, value in obj.items()
            if isinstance(value, list)
        ]

        if len(list_members) == 1:
            key, value = list_members[0]
            return len(value), f"list inside '{key}'"

    # Structure se record count safely determine nahi hua
    return None, "requires structure inspection"


for file_path in json_files:

    with file_path.open("r", encoding="utf-8") as file:
        obj = json.load(file)

    expected_rows = EXPECTED_COUNTS[file_path.name]
    actual_rows, count_method = inspect_json_records(obj)

    if isinstance(obj, dict):
        json_keys = list(obj.keys())
        list_members = {
            key: len(value)
            for key, value in obj.items()
            if isinstance(value, list)
        }
    elif isinstance(obj, list):
        json_keys = (
            list(obj[0].keys())
            if obj and isinstance(obj[0], dict)
            else []
        )
        list_members = {}
    else:
        json_keys = []
        list_members = {}

    count_matches = (
        actual_rows == expected_rows
        if actual_rows is not None
        else None
    )

    audit_records.append({
        "file_name": file_path.name,
        "format": "JSON",
        "actual_records": actual_rows,
        "expected_records": expected_rows,
        "count_matches": count_matches,
        "column_count": len(json_keys),
        "blank_cells": None,
        "fully_duplicate_rows": None,
        "json_keys": ", ".join(map(str, json_keys)),
        "json_list_members": json.dumps(list_members),
    })

    print(f"\nFile: {file_path.name}")
    print(f"Top-level type: {type(obj).__name__}")
    print(f"Expected records: {expected_rows:,}")
    print(f"Observed records: {actual_rows}")
    print(f"Counting method: {count_method}")
    print(f"Top-level keys / record fields: {json_keys}")
    print(f"Lists inside JSON: {list_members}")
    print(f"Count matches: {count_matches}")

# --------------------------------------------------
# PART 3: SAVE AUDIT REPORT
# --------------------------------------------------

audit_df = pd.DataFrame(audit_records)

audit_path = DOCS_DIR / "source_audit.csv"
audit_df.to_csv(audit_path, index=False)

print("\n" + "=" * 65)
print("AUDIT SUMMARY")
print("=" * 65)

print(
    audit_df[
        [
            "file_name",
            "actual_records",
            "expected_records",
            "count_matches",
        ]
    ].to_string(index=False)
)

print("\nTotal files inspected:", len(audit_df))
print("Audit report saved to:", audit_path)

print(
    "\nCount mismatches:",
    int((audit_df["count_matches"] == False).sum())
)


PEOPLEOPS HACKATHON - SOURCE DATA AUDIT

CSV files found: 10

File: applications.csv
Rows: 3,835
Expected rows: 3,835
Count matches: True
Columns (6): ['application_id', 'candidate_id', 'job_req_id', 'dept_id', 'stage', 'stage_date']
Blank cells: 0
Fully duplicated rows: 0

File: biometric_punches.csv
Rows: 345,280
Expected rows: 345,280
Count matches: True
Columns (4): ['punch_id', 'device_id', 'device_user_id', 'punch_time']
Blank cells: 0
Fully duplicated rows: 0

File: departments.csv
Rows: 25
Expected rows: 25
Count matches: True
Columns (6): ['dept_id', 'dept_name', 'plant', 'cost_center', 'previous_name', 'renamed_on']
Blank cells: 46
Fully duplicated rows: 0

File: device_mapping.csv
Rows: 1,260
Expected rows: 1,260
Count matches: True
Columns (4): ['device_id', 'device_user_id', 'emp_id', 'enrolled_on']
Blank cells: 0
Fully duplicated rows: 0

File: employees.csv
Rows: 1,316
Expected rows: 1,316
Count matches: True
Columns (16): ['emp_id', 'full_name', 'gender', 'date_of_birth

In [6]:
%%writefile /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/ingestion/load_raw.py

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sqlite3
import pandas as pd


# --------------------------------------------------
# 1. PROJECT SETTINGS
# --------------------------------------------------

PROJECT_DIR = Path(__file__).resolve().parents[1]
RAW_DIR = PROJECT_DIR / "data" / "raw"
DOCS_DIR = PROJECT_DIR / "docs"

DB_PATH = PROJECT_DIR / "data" / "peopleops.sqlite"
REPORT_PATH = DOCS_DIR / "raw_load_audit.csv"

EXPECTED_COUNTS = {
    "employees.csv": 1316,
    "employment_history.csv": 3686,
    "departments.csv": 25,
    "shifts.json": 4,
    "shift_roster.csv": 36402,
    "device_mapping.csv": 1260,
    "biometric_punches.csv": 345280,
    "holidays.csv": 12,
    "leave_requests.json": 5372,
    "overtime_requests.csv": 22098,
    "payroll_lines.csv": 53617,
    "applications.csv": 3835,
}

METADATA_COLUMNS = {
    "_source_file",
    "_loaded_at",
    "_row_hash",
    "_source_row_number",
    "_raw_record_json",
}


# --------------------------------------------------
# 2. HELPER FUNCTIONS
# --------------------------------------------------

def quote_identifier(name):
    """Safely quote a SQLite table or column name."""
    return '"' + name.replace('"', '""') + '"'


def get_table_name(file_path):
    """Example: employees.csv becomes raw_employees."""
    return "raw_" + file_path.stem.lower()


def validate_source_columns(columns):
    """Prevent source fields from overwriting metadata."""
    duplicates = set(columns) & METADATA_COLUMNS

    if duplicates:
        raise ValueError(
            f"Source columns conflict with metadata: {duplicates}"
        )

    if len(columns) != len(set(columns)):
        raise ValueError("Duplicate source column names detected.")


def get_json_records(file_path):
    """
    Read JSON without flattening nested objects.
    Each top-level business record becomes one raw row.
    """
    with file_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if isinstance(payload, list):
        records = payload

    elif isinstance(payload, dict):
        # Prefer the collection belonging to this dataset.
        preferred_keys = {
            "shifts.json": ["shifts"],
            "leave_requests.json": [
                "leave_requests",
                "requests",
                "data",
                "records",
            ],
        }

        records = None

        for key in preferred_keys.get(file_path.name, []):
            if isinstance(payload.get(key), list):
                records = payload[key]
                break

        # Fallback: accept a single list at the top level.
        if records is None:
            lists = [
                value
                for value in payload.values()
                if isinstance(value, list)
            ]

            if len(lists) == 1:
                records = lists[0]

        if records is None:
            raise ValueError(
                f"Could not identify the record list in "
                f"{file_path.name}. Inspect its JSON structure first."
            )

    else:
        raise ValueError(
            f"Unexpected JSON structure in {file_path.name}"
        )

    # Keep non-dictionary records without discarding them.
    return [
        record if isinstance(record, dict) else {"value": record}
        for record in records
    ]


def json_value_to_text(value):
    """Represent each JSON field as text without flattening it."""
    if value is None:
        return None

    if isinstance(value, (dict, list)):
        return json.dumps(
            value,
            ensure_ascii=False,
            separators=(",", ":"),
        )

    if isinstance(value, bool):
        return "true" if value else "false"

    return str(value)


def create_raw_table(
    connection,
    table_name,
    source_columns,
    is_json,
):
    """Create a raw table and verify its column structure."""
    metadata = [
        "_source_file",
        "_loaded_at",
        "_row_hash",
        "_source_row_number",
    ]

    if is_json:
        metadata.append("_raw_record_json")

    all_columns = source_columns + metadata

    definitions = [
        f"{quote_identifier(column)} TEXT"
        for column in source_columns
    ]

    for column in metadata:
        definitions.append(
            f"{quote_identifier(column)} TEXT NOT NULL"
        )

    definitions.append(
        "UNIQUE "
        '("_source_file", "_source_row_number")'
    )

    create_sql = (
        f"CREATE TABLE IF NOT EXISTS "
        f"{quote_identifier(table_name)} "
        f"({', '.join(definitions)})"
    )

    connection.execute(create_sql)

    actual_columns = [
        row[1]
        for row in connection.execute(
            f"PRAGMA table_info({quote_identifier(table_name)})"
        )
    ]

    if actual_columns != all_columns:
        raise RuntimeError(
            f"Schema mismatch in {table_name}.\n"
            f"Expected: {all_columns}\n"
            f"Found: {actual_columns}"
        )

    return all_columns


def make_upsert_sql(table_name, source_columns, is_json):
    """Insert source rows; update existing source positions on rerun."""
    metadata = [
        "_source_file",
        "_loaded_at",
        "_row_hash",
        "_source_row_number",
    ]

    if is_json:
        metadata.append("_raw_record_json")

    columns = source_columns + metadata

    column_sql = ", ".join(
        quote_identifier(column) for column in columns
    )
    placeholders = ", ".join("?" for _ in columns)

    update_columns = source_columns + [
        "_loaded_at",
        "_row_hash",
    ]

    if is_json:
        update_columns.append("_raw_record_json")

    update_sql = ", ".join(
        f"{quote_identifier(column)} = "
        f"excluded.{quote_identifier(column)}"
        for column in update_columns
    )

    return f"""
        INSERT INTO {quote_identifier(table_name)}
            ({column_sql})
        VALUES ({placeholders})
        ON CONFLICT ("_source_file", "_source_row_number")
        DO UPDATE SET {update_sql}
    """


# --------------------------------------------------
# 3. INGEST CSV FILES
# --------------------------------------------------

def load_csv(connection, file_path):
    table_name = get_table_name(file_path)

    header_df = pd.read_csv(
        file_path,
        dtype="string",
        keep_default_na=False,
        nrows=0,
    )

    columns = header_df.columns.tolist()
    validate_source_columns(columns)

    create_raw_table(
        connection,
        table_name,
        columns,
        is_json=False,
    )

    insert_sql = make_upsert_sql(
        table_name,
        columns,
        is_json=False,
    )

    # Refresh only this source file's table rows.
    # The full source file remains unchanged on disk.
    connection.execute(
        f"DELETE FROM {quote_identifier(table_name)} "
        'WHERE "_source_file" = ?',
        (file_path.name,),
    )

    loaded_at = datetime.now(timezone.utc).isoformat()
    source_rows = 0
    batch = []
    batch_size = 20000

    for chunk in pd.read_csv(
        file_path,
        dtype="string",
        keep_default_na=False,
        chunksize=batch_size,
    ):
        for row in chunk.itertuples(index=False, name=None):
            source_rows += 1

            values = [
                "" if pd.isna(value) else str(value)
                for value in row
            ]

            hash_input = json.dumps(
                values,
                ensure_ascii=False,
                separators=(",", ":"),
            )

            row_hash = hashlib.sha256(
                hash_input.encode("utf-8")
            ).hexdigest()

            batch.append(
                values
                + [
                    file_path.name,
                    loaded_at,
                    row_hash,
                    str(source_rows),
                ]
            )

            if len(batch) >= batch_size:
                connection.executemany(insert_sql, batch)
                batch.clear()

    if batch:
        connection.executemany(insert_sql, batch)

    table_rows = connection.execute(
        f"SELECT COUNT(*) FROM {quote_identifier(table_name)}"
    ).fetchone()[0]

    return source_rows, table_name, table_rows


# --------------------------------------------------
# 4. INGEST JSON FILES
# --------------------------------------------------

def load_json(connection, file_path):
    table_name = get_table_name(file_path)
    records = get_json_records(file_path)

    # Preserve every source field; nested lists and objects
    # are stored as JSON text, not flattened.
    columns = list(
        dict.fromkeys(
            key
            for record in records
            for key in record.keys()
        )
    )

    validate_source_columns(columns)

    create_raw_table(
        connection,
        table_name,
        columns,
        is_json=True,
    )

    insert_sql = make_upsert_sql(
        table_name,
        columns,
        is_json=True,
    )

    connection.execute(
        f"DELETE FROM {quote_identifier(table_name)} "
        'WHERE "_source_file" = ?',
        (file_path.name,),
    )

    loaded_at = datetime.now(timezone.utc).isoformat()
    batch = []
    batch_size = 20000

    for row_number, record in enumerate(records, start=1):
        raw_record_json = json.dumps(
            record,
            ensure_ascii=False,
            separators=(",", ":"),
        )

        row_hash = hashlib.sha256(
            raw_record_json.encode("utf-8")
        ).hexdigest()

        values = [
            json_value_to_text(record.get(column))
            for column in columns
        ]

        batch.append(
            values
            + [
                file_path.name,
                loaded_at,
                row_hash,
                str(row_number),
                raw_record_json,
            ]
        )

        if len(batch) >= batch_size:
            connection.executemany(insert_sql, batch)
            batch.clear()

    if batch:
        connection.executemany(insert_sql, batch)

    table_rows = connection.execute(
        f"SELECT COUNT(*) FROM {quote_identifier(table_name)}"
    ).fetchone()[0]

    return len(records), table_name, table_rows


# --------------------------------------------------
# 5. RUN THE RAW INGESTION PIPELINE
# --------------------------------------------------

def main():
    if not RAW_DIR.exists():
        raise FileNotFoundError(
            f"Raw data folder not found: {RAW_DIR}"
        )

    DOCS_DIR.mkdir(parents=True, exist_ok=True)

    source_files = (
        sorted(RAW_DIR.glob("*.csv"))
        + sorted(RAW_DIR.glob("*.json"))
    )

    if len(source_files) != 12:
        raise RuntimeError(
            f"Expected 12 source files, found {len(source_files)}"
        )

    print("=" * 65)
    print("PEOPLEOPS - RAW INGESTION")
    print("=" * 65)
    print("Database:", DB_PATH)

    report_rows = []

    connection = sqlite3.connect(DB_PATH)

    try:
        connection.execute("PRAGMA journal_mode=WAL")
        connection.execute("PRAGMA synchronous=NORMAL")

        for file_path in source_files:

            if file_path.name not in EXPECTED_COUNTS:
                raise ValueError(
                    f"No expected count defined for {file_path.name}"
                )

            print(f"\nLoading: {file_path.name}")

            if file_path.suffix.lower() == ".csv":
                source_rows, table_name, table_rows = load_csv(
                    connection,
                    file_path,
                )
            else:
                source_rows, table_name, table_rows = load_json(
                    connection,
                    file_path,
                )

            expected_rows = EXPECTED_COUNTS[file_path.name]

            source_matches_table = source_rows == table_rows
            source_matches_expected = source_rows == expected_rows

            report_rows.append({
                "source_file": file_path.name,
                "table_name": table_name,
                "source_rows": source_rows,
                "table_rows": table_rows,
                "expected_rows_from_pdf": expected_rows,
                "source_matches_table": source_matches_table,
                "matches_pdf_count": source_matches_expected,
            })

            print("Table:", table_name)
            print(f"Source rows: {source_rows:,}")
            print(f"Database rows: {table_rows:,}")
            print("Source and table match:", source_matches_table)
            print("PDF expected count matches:", source_matches_expected)

        connection.commit()

    except Exception:
        connection.rollback()
        raise

    finally:
        connection.close()

    report_df = pd.DataFrame(report_rows)
    report_df.to_csv(REPORT_PATH, index=False)

    failures = report_df[
        (~report_df["source_matches_table"])
        | (~report_df["matches_pdf_count"])
    ]

    print("\n" + "=" * 65)
    print("RAW INGESTION SUMMARY")
    print("=" * 65)

    print(
        report_df[
            [
                "source_file",
                "source_rows",
                "table_rows",
                "source_matches_table",
                "matches_pdf_count",
            ]
        ].to_string(index=False)
    )

    print("\nTotal files loaded:", len(report_df))
    print("Failed count checks:", len(failures))
    print("SQLite database saved to:", DB_PATH)
    print("Load audit saved to:", REPORT_PATH)


if __name__ == "__main__":
    main()

Overwriting /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/ingestion/load_raw.py


In [7]:
from pathlib import Path
import subprocess
import sys

project_dir = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

script_path = project_dir / "ingestion" / "load_raw.py"

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print("\n===== ERROR DETAILS =====")
    print(result.stderr)

print("\nExit code:", result.returncode)

PEOPLEOPS - RAW INGESTION
Database: /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/data/peopleops.sqlite

Loading: applications.csv
Table: raw_applications
Source rows: 3,835
Database rows: 3,835
Source and table match: True
PDF expected count matches: True

Loading: biometric_punches.csv
Table: raw_biometric_punches
Source rows: 345,280
Database rows: 345,280
Source and table match: True
PDF expected count matches: True

Loading: departments.csv
Table: raw_departments
Source rows: 25
Database rows: 25
Source and table match: True
PDF expected count matches: True

Loading: device_mapping.csv
Table: raw_device_mapping
Source rows: 1,260
Database rows: 1,260
Source and table match: True
PDF expected count matches: True

Loading: employees.csv
Table: raw_employees
Source rows: 1,316
Database rows: 1,316
Source and table match: True
PDF expected count matches: True

Loading: employment_history.csv
Table: raw_employment_history
Source rows: 3,686
Database rows: 3,686
Source and tabl

In [8]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

PROJECT = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

DB_PATH = PROJECT / "data" / "peopleops.sqlite"
AUDIT_PATH = PROJECT / "docs" / "raw_load_audit.csv"

# ------------------------------------
# 1. Verify all raw-load counts
# ------------------------------------
audit = pd.read_csv(AUDIT_PATH)

print("===== RAW LOAD VERIFICATION =====")
print(
    audit[
        [
            "source_file",
            "source_rows",
            "table_rows",
            "source_matches_table",
            "matches_pdf_count",
        ]
    ].to_string(index=False)
)

failed = audit[
    (~audit["source_matches_table"].astype(bool))
    | (~audit["matches_pdf_count"].astype(bool))
]

print("\nFiles loaded:", len(audit))
print("Count failures:", len(failed))

# ------------------------------------
# 2. Read the raw punches and mappings
# ------------------------------------
with sqlite3.connect(DB_PATH) as conn:
    punches = pd.read_sql_query(
        """
        SELECT device_id, device_user_id, punch_time
        FROM raw_biometric_punches
        """,
        conn,
    )

    mapping = pd.read_sql_query(
        """
        SELECT device_id, device_user_id, emp_id
        FROM raw_device_mapping
        """,
        conn,
    )

print("\nRaw punch records:", len(punches))
print("Device mappings:", len(mapping))

key_cols = ["device_id", "device_user_id"]

mapping_duplicates = mapping.duplicated(
    subset=key_cols,
    keep=False,
).sum()

print("Duplicate mapping-key rows:", mapping_duplicates)

# Use one mapping per device/user pair for diagnostic only.
# If conflicting mappings exist, they must be investigated separately.
mapping_unique = mapping.drop_duplicates(
    subset=key_cols,
    keep="last",
)

mapped = punches.merge(
    mapping_unique,
    on=key_cols,
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Mapped punch records:", int((mapped["_merge"] == "both").sum()))
print("Unmapped punch records:", int((mapped["_merge"] == "left_only").sum()))

mapped["punch_time"] = pd.to_datetime(
    mapped["punch_time"],
    errors="coerce",
)

invalid_times = int(mapped["punch_time"].isna().sum())
print("Invalid punch timestamps:", invalid_times)

valid = mapped.loc[
    (mapped["_merge"] == "both")
    & mapped["punch_time"].notna()
].copy()

# ------------------------------------
# 3. Compare punch times with shift boundaries
# ------------------------------------
# Shift boundaries from the supplied data dictionary:
# DAY: 06:00-14:00
# EVE: 14:00-22:00
# NIGHT: 22:00-06:00
# GEN: 09:00-17:30

boundaries = np.array([360, 540, 840, 1050, 1320])

minute_of_day = (
    valid["punch_time"].dt.hour.to_numpy() * 60
    + valid["punch_time"].dt.minute.to_numpy()
    + valid["punch_time"].dt.second.to_numpy() / 60
)

valid["minute_of_day"] = minute_of_day

def median_boundary_distance(minutes, correction):
    corrected = (minutes + correction) % 1440

    difference = np.abs(
        corrected[:, None] - boundaries[None, :]
    )

    circular_difference = np.minimum(
        difference,
        1440 - difference,
    )

    nearest_boundary_distance = circular_difference.min(axis=1)

    return float(np.median(nearest_boundary_distance))


results = []

for device_id, group in valid.groupby("device_id"):
    times = group["minute_of_day"].to_numpy()

    median_unchanged = median_boundary_distance(times, 0)

    candidates = {
        offset: median_boundary_distance(times, offset)
        for offset in [-60, 0, 60]
    }

    best_offset = min(
        candidates,
        key=candidates.get,
    )

    results.append({
        "device_id": device_id,
        "mapped_punches": len(group),
        "median_distance_unchanged_min": round(
            median_unchanged, 2
        ),
        "best_tested_correction_min": best_offset,
        "median_distance_after_test_min": round(
            candidates[best_offset], 2
        ),
        "improvement_min": round(
            median_unchanged - candidates[best_offset], 2
        ),
    })

diagnostic = pd.DataFrame(results).sort_values(
    "improvement_min",
    ascending=False,
)

print("\n===== DEVICE CLOCK DIAGNOSTIC =====")
print(diagnostic.to_string(index=False))

diagnostic_path = PROJECT / "docs" / "device_clock_diagnostic.csv"
diagnostic.to_csv(diagnostic_path, index=False)

print("\nDiagnostic report saved to:", diagnostic_path)
print(
    "\nNote: this is an initial diagnostic, not proof of a clock error."
)

===== RAW LOAD VERIFICATION =====
           source_file  source_rows  table_rows  source_matches_table  matches_pdf_count
      applications.csv         3835        3835                  True               True
 biometric_punches.csv       345280      345280                  True               True
       departments.csv           25          25                  True               True
    device_mapping.csv         1260        1260                  True               True
         employees.csv         1316        1316                  True               True
employment_history.csv         3686        3686                  True               True
          holidays.csv           12          12                  True               True
 overtime_requests.csv        22098       22098                  True               True
     payroll_lines.csv        53617       53617                  True               True
      shift_roster.csv        36402       36402                  True       

In [9]:
from pathlib import Path
import pandas as pd

project = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

diagnostic_file = (
    project / "docs" / "device_clock_diagnostic.csv"
)

diagnostic = pd.read_csv(diagnostic_file)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(diagnostic)

,device_id,mapped_punches,median_distance_unchanged_min,best_tested_correction_min,median_distance_after_test_min,improvement_min
0,DEV-SITE-01,93248,40.43,-60,31.26,9.18
1,DEV-FSD-01,103280,12.05,0,12.05,0.00
2,DEV-HO-01,34784,11.48,0,11.48,0.00
3,DEV-KOR-01,107006,11.92,0,11.92,0.00


In [10]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT = Path.home() / "Documents" / "indus-weave-hr-pipeline"
DB_PATH = PROJECT / "data" / "peopleops.sqlite"

with sqlite3.connect(DB_PATH) as conn:
    punches = pd.read_sql_query(
        """
        SELECT punch_id, device_id, device_user_id, punch_time
        FROM raw_biometric_punches
        """,
        conn,
    )

    mapping = pd.read_sql_query(
        """
        SELECT device_id, device_user_id, emp_id
        FROM raw_device_mapping
        """,
        conn,
    )

    roster = pd.read_sql_query(
        """
        SELECT emp_id, week_start, shift_code
        FROM raw_shift_roster
        """,
        conn,
    )

# Map each biometric device user to an employee
punches = punches.merge(
    mapping,
    on=["device_id", "device_user_id"],
    how="left",
    validate="many_to_one",
)

punches["punch_time"] = pd.to_datetime(
    punches["punch_time"], errors="coerce"
)

punches = punches.dropna(
    subset=["emp_id", "punch_time"]
).copy()

# Keep a reproducible sample so this check runs faster
samples = []

for device, group in punches.groupby("device_id"):
    samples.append(
        group.sample(
            n=min(10000, len(group)),
            random_state=42,
        )
    )

sample = pd.concat(samples, ignore_index=True)
sample["sample_id"] = range(len(sample))

# Prepare weekly roster and carry the previous shift forward
roster["week_start"] = pd.to_datetime(
    roster["week_start"], errors="coerce"
)

roster = (
    roster.dropna(subset=["week_start"])
    .sort_values(["emp_id", "week_start"])
    .drop_duplicates(["emp_id", "week_start"], keep="last")
)

weeks = pd.date_range(
    roster["week_start"].min(),
    roster["week_start"].max(),
    freq="W-MON",
)

employees = roster["emp_id"].drop_duplicates().tolist()

weekly = pd.MultiIndex.from_product(
    [employees, weeks],
    names=["emp_id", "week_start"],
).to_frame(index=False)

weekly = weekly.merge(
    roster,
    on=["emp_id", "week_start"],
    how="left",
)

weekly = weekly.sort_values(["emp_id", "week_start"])
weekly["shift_code"] = (
    weekly.groupby("emp_id")["shift_code"].ffill()
)

# Actual shift boundaries from shifts.json
shift_minutes = {
    "DAY": (360, 840),    # 06:00 - 14:00
    "EVE": (840, 1320),   # 14:00 - 22:00
    "NIGHT": (1320, 360), # 22:00 - next day 06:00
    "GEN": (540, 1050),   # 09:00 - 17:30
}

scores = []

for correction in [-60, 0, 60]:
    base = sample[
        ["sample_id", "device_id", "emp_id", "punch_time"]
    ].copy()

    base["adjusted_time"] = (
        base["punch_time"]
        + pd.to_timedelta(correction, unit="m")
    )

    candidates = []

    # Consider the punch date and the previous date.
    # This is needed for night-shift check-outs.
    for prior_day in [0, 1]:
        candidate = base.copy()

        candidate["shift_date"] = (
            candidate["adjusted_time"].dt.normalize()
            - pd.to_timedelta(prior_day, unit="D")
        )

        candidate["week_start"] = (
            candidate["shift_date"]
            - pd.to_timedelta(
                candidate["shift_date"].dt.weekday,
                unit="D",
            )
        )

        candidate = candidate.merge(
            weekly[["emp_id", "week_start", "shift_code"]],
            on=["emp_id", "week_start"],
            how="left",
        )

        candidate = candidate.dropna(subset=["shift_code"])

        candidate["start_min"] = candidate["shift_code"].map(
            {key: value[0] for key, value in shift_minutes.items()}
        )

        candidate["end_min"] = candidate["shift_code"].map(
            {key: value[1] for key, value in shift_minutes.items()}
        )

        candidate["shift_start"] = (
            candidate["shift_date"]
            + pd.to_timedelta(candidate["start_min"], unit="m")
        )

        candidate["shift_end"] = (
            candidate["shift_date"]
            + pd.to_timedelta(candidate["end_min"], unit="m")
        )

        overnight = (
            candidate["end_min"] <= candidate["start_min"]
        )

        candidate.loc[overnight, "shift_end"] += pd.Timedelta(days=1)

        start_distance = (
            candidate["adjusted_time"] - candidate["shift_start"]
        ).abs().dt.total_seconds() / 60

        end_distance = (
            candidate["adjusted_time"] - candidate["shift_end"]
        ).abs().dt.total_seconds() / 60

        candidate["distance_min"] = pd.concat(
            [start_distance, end_distance], axis=1
        ).min(axis=1)

        candidates.append(
            candidate[["sample_id", "device_id", "distance_min"]]
        )

    if candidates:
        candidate_df = pd.concat(candidates, ignore_index=True)

        # Each punch uses the nearest valid shift boundary.
        nearest = candidate_df.groupby(
            ["sample_id", "device_id"]
        )["distance_min"].min().reset_index()

        result = (
            nearest.groupby("device_id")["distance_min"]
            .median()
            .reset_index(name="median_boundary_distance")
        )

        result["correction_minutes_tested"] = correction
        result["sampled_punches"] = nearest.groupby(
            "device_id"
        ).size().reindex(result["device_id"]).to_numpy()

        scores.append(result)

score_df = pd.concat(scores, ignore_index=True)

pivot = score_df.pivot(
    index="device_id",
    columns="correction_minutes_tested",
    values="median_boundary_distance",
)

pivot.columns = [
    f"median_distance_if_correction_{int(column)}_min"
    for column in pivot.columns
]

print("===== ROSTER-BASED CLOCK DIAGNOSTIC =====")
display(pivot.round(2))

print(
    "\nInterpretation: lower median distance means a closer "
    "match to scheduled shift boundaries in this sample."
)

print(
    "This is evidence for investigation, not a final proof "
    "of a clock fault."
)

/var/folders/k5/b2ytkmms5pbfvwwcnbx0ddf40000gn/T/ipykernel_1866/201027830.py:133: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  candidate["shift_date"].dt.weekday,
/var/folders/k5/b2ytkmms5pbfvwwcnbx0ddf40000gn/T/ipykernel_1866/201027830.py:133: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  candidate["shift_date"].dt.weekday,
/var/folders/k5/b2ytkmms5pbfvwwcnbx0ddf40000gn/T/ipykernel_1866/201027830.py:133: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  candidate["shift_date"].dt.weekday,
/var/folders/k5/b2ytkmms5pbfvwwcnbx0ddf40000gn/T/ipykernel_1866/201027830.py:133: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  candidate["shift_date"].dt.weekday,
/var/folders/k5/b2ytkmms

===== ROSTER-BASED CLOCK DIAGNOSTIC =====


,median_distance_if_correction_-60_min,median_distance_if_correction_0_min,median_distance_if_correction_60_min
device_id,,,
DEV-FSD-01,55.62,12.20,65.20
DEV-HO-01,55.80,11.55,64.58
DEV-KOR-01,55.71,12.18,65.02
DEV-SITE-01,35.89,41.93,101.36



Interpretation: lower median distance means a closer match to scheduled shift boundaries in this sample.
This is evidence for investigation, not a final proof of a clock fault.


In [11]:
from pathlib import Path
import shutil
import subprocess
import sys

project = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

downloaded_file = (
    Path.home()
    / "Downloads"
    / "stage2_clean_and_attendance.py"
)

target_file = (
    project
    / "transform"
    / "stage2_clean_and_attendance.py"
)

if not downloaded_file.exists():
    raise FileNotFoundError(
        "Downloaded script nahi mili. "
        "Mac ke Downloads folder mein file locate karo."
    )

shutil.copy2(downloaded_file, target_file)

print("Script copied successfully.")
print("Running cleaning and attendance pipeline...\n")

result = subprocess.run(
    [sys.executable, str(target_file)],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print("\n===== ERROR DETAILS =====")
    print(result.stderr)
else:
    print("\nPipeline completed. Exit code:", result.returncode)

Script copied successfully.
Running cleaning and attendance pipeline...

PEOPLEOPS STAGE 2 + STAGE 3: CLEANING AND ATTENDANCE
Database: /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/data/peopleops.sqlite


===== ERROR DETAILS =====
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandas/io/sql.py", line 2671, in _execute_insert
    conn.executemany(self.insert_statement(num_rows=1), data_list)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
sqlite3.ProgrammingError: Error binding parameter 21: type 'Timestamp' is not supported

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/abdullahahmed/Documents/indus-weave-hr-pipeline/transform/stage2_clean_and_attendance.py", line 629, in <module>
    main()
    ~~~~^^
  File "/Users/abdullahahmed/Documents/indus-weave-hr-pipeline/transform/stage2_clean_and_attendance.py", lin

In [12]:
from pathlib import Path
import re
import subprocess
import sys

project = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

script_path = (
    project
    / "transform"
    / "stage2_clean_and_attendance.py"
)

source = script_path.read_text(encoding="utf-8")

# Replace the old write_table function with a safe version.
pattern = (
    r"def write_table\(conn, name, df\):"
    r".*?"
    r"(?=\n\ndef read_raw\(conn, table\):)"
)

replacement = '''
def _sqlite_safe_value(value):
    """Convert pandas/Python values into SQLite-compatible values."""
    import numpy as np

    if value is None:
        return None

    if isinstance(value, pd.Timestamp):
        return None if pd.isna(value) else value.isoformat(sep=" ")

    if isinstance(value, datetime):
        return value.isoformat(sep=" ")

    if isinstance(value, date):
        return value.isoformat()

    if isinstance(value, pd.Timedelta):
        return str(value)

    try:
        missing = pd.isna(value)
        if isinstance(missing, (bool, np.bool_)) and missing:
            return None
    except (TypeError, ValueError):
        pass

    # Convert NumPy scalar values into standard Python scalars.
    if isinstance(value, np.generic):
        value = value.item()

    return value


def write_table(conn, name, df):
    """Write a DataFrame to SQLite using safe scalar values."""
    out = df.copy()

    for column in out.columns:
        out[column] = out[column].map(_sqlite_safe_value)

    out.to_sql(
        name,
        conn,
        if_exists="replace",
        index=False,
        chunksize=20000,
    )
'''

fixed_source, replacements = re.subn(
    pattern,
    replacement.strip(),
    source,
    count=1,
    flags=re.DOTALL,
)

if replacements != 1:
    raise RuntimeError(
        "Function replacement nahi hua. "
        "Script ko manually change mat karo; error share karo."
    )

# Check the fixed script syntax before running.
compile(fixed_source, str(script_path), "exec")

script_path.write_text(fixed_source, encoding="utf-8")

print("SQLite Timestamp fix applied.")
print("Python syntax check: PASSED")
print("\nRerunning Stage 2 + Stage 3...\n")

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print("\n===== ERROR DETAILS =====")
    print(result.stderr[-12000:])

print("\nExit code:", result.returncode)


SQLite Timestamp fix applied.
Python syntax check: PASSED

Rerunning Stage 2 + Stage 3...

PEOPLEOPS STAGE 2 + STAGE 3: CLEANING AND ATTENDANCE
Database: /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/data/peopleops.sqlite

STAGING + ATTENDANCE SUMMARY
employees_raw: 1316
employees_clean: 1266
employee_duplicate_rows_logged: 50
punches_raw: 345280
mapped_punches_clean: 322143
unmapped_punches_rejected: 6962
near_duplicate_punches_logged: 16175
approved_leave_requests: 4282
approved_leave_days: 8604
attendance_daily_rows: 217584
negative_worked_hours: 0
clock_correction_device: DEV-SITE-01
clock_correction_minutes: -60

Created tables: stg_* (raw_* tables untouched)
Reports: /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/docs/staging_reconciliation.csv
          /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/docs/clock_correction_evidence.csv


Exit code: 0


In [13]:
from pathlib import Path
import shutil
import subprocess
import sys

project = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

downloaded = (
    Path.home()
    / "Downloads"
    / "stage4_warehouse_analytics.py"
)

target = (
    project
    / "warehouse"
    / "stage4_warehouse_analytics.py"
)

if not downloaded.exists():
    raise FileNotFoundError(
        "Downloads folder mein stage4_warehouse_analytics.py nahi mili. "
        "Downloaded file ka exact naam check karo."
    )

shutil.copy2(downloaded, target)

print("Stage 4-6 script copied.")
print("Running warehouse, analytics and data-quality checks...\n")

result = subprocess.run(
    [sys.executable, str(target)],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print("\n===== ERROR DETAILS =====")
    print(result.stderr[-12000:])

print("\nExit code:", result.returncode)

Stage 4-6 script copied.
Running warehouse, analytics and data-quality checks...



===== ERROR DETAILS =====
Traceback (most recent call last):
  File "/Users/abdullahahmed/Documents/indus-weave-hr-pipeline/warehouse/stage4_warehouse_analytics.py", line 1323, in <module>
    main()
    ~~~~^^
  File "/Users/abdullahahmed/Documents/indus-weave-hr-pipeline/warehouse/stage4_warehouse_analytics.py", line 1229, in main
    create_fact_tables_and_load(conn)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/Users/abdullahahmed/Documents/indus-weave-hr-pipeline/warehouse/stage4_warehouse_analytics.py", line 511, in create_fact_tables_and_load
    raise RuntimeError(f"{missing_att_emp} attendance rows have no SCD employee version. Review history dates.")
RuntimeError: 181 attendance rows have no SCD employee version. Review history dates.


Exit code: 1


In [14]:
from pathlib import Path
import sqlite3
import pandas as pd

project = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)
db_path = project / "data" / "peopleops.sqlite"

with sqlite3.connect(db_path) as conn:
    # Identify the actual date column in attendance.
    columns = {
        row[1]
        for row in conn.execute(
            'PRAGMA table_info("stg_attendance_daily")'
        )
    }

    date_column = (
        "work_date_key"
        if "work_date_key" in columns
        else "work_date"
    )

    query = f"""
    WITH attendance AS (
        SELECT
            a.emp_id,
            COALESCE(NULLIF(a."{date_column}", ''), a.work_date)
                AS work_date
        FROM stg_attendance_daily a
        WHERE NOT EXISTS (
            SELECT 1
            FROM dim_employee e
            WHERE e.emp_id = a.emp_id
              AND e.valid_from <=
                  COALESCE(NULLIF(a."{date_column}", ''), a.work_date)
        )
    ),
    first_history AS (
        SELECT emp_id, MIN(valid_from) AS first_history_from
        FROM stg_employment_history
        GROUP BY emp_id
    )
    SELECT
        a.emp_id,
        COUNT(*) AS attendance_rows,
        MIN(a.work_date) AS first_attendance_date,
        MAX(a.work_date) AS last_attendance_date,
        p.hire_date,
        h.first_history_from,
        CASE
            WHEN p.hire_date IS NOT NULL
             AND MIN(a.work_date) < p.hire_date
                THEN 'Attendance before hire date'
            WHEN h.first_history_from IS NOT NULL
                THEN 'Before first employment-history record'
            ELSE 'No matching employment-history version'
        END AS possible_reason
    FROM attendance a
    LEFT JOIN stg_employees p ON p.emp_id = a.emp_id
    LEFT JOIN first_history h ON h.emp_id = a.emp_id
    GROUP BY
        a.emp_id, p.hire_date, h.first_history_from
    ORDER BY attendance_rows DESC, a.emp_id
    """

    exceptions = pd.read_sql_query(query, conn)

print("UNMATCHED ATTENDANCE DIAGNOSTIC")
print("Affected employees:", len(exceptions))
print(
    "Affected attendance rows:",
    int(exceptions["attendance_rows"].sum())
)
display(exceptions)

UNMATCHED ATTENDANCE DIAGNOSTIC
Affected employees: 4
Affected attendance rows: 181


,emp_id,attendance_rows,first_attendance_date,last_attendance_date,hire_date,first_history_from,possible_reason
0,E10168,60,2026-07-09,2026-09-06,2026-07-09,2026-07-09,Before first employment-history record
1,E10160,51,2026-08-11,2026-09-30,2026-08-11,2026-08-11,Before first employment-history record
2,E10905,50,2026-08-12,2026-09-30,2026-08-12,2026-08-12,Before first employment-history record
3,E10341,20,2026-09-11,2026-09-30,2026-09-11,2026-09-11,Before first employment-history record


In [15]:
from pathlib import Path
import sqlite3
import pandas as pd

db = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
    / "data"
    / "peopleops.sqlite"
)

ids = ["E10168", "E10160", "E10905", "E10341"]

with sqlite3.connect(db) as conn:
    print("=== DIMENSION DATE RANGES ===")
    display(pd.read_sql_query(
        """
        SELECT emp_id, valid_from, valid_to, dept_id
        FROM dim_employee
        WHERE emp_id IN (?, ?, ?, ?)
        ORDER BY emp_id, valid_from
        """,
        conn, params=ids
    ))

    print("=== ATTENDANCE DATE VALUES ===")
    display(pd.read_sql_query(
        """
        SELECT emp_id, work_date_key, work_date
        FROM stg_attendance_daily
        WHERE emp_id IN (?, ?, ?, ?)
        ORDER BY emp_id, work_date_key
        LIMIT 20
        """,
        conn, params=ids
    ))

=== DIMENSION DATE RANGES ===


,emp_id,valid_from,valid_to,dept_id
0,E10160,2026-11-08,9999-12-31,D14
1,E10168,2026-09-07,9999-12-31,D18
2,E10341,2026-11-09,9999-12-31,D20
3,E10905,2026-12-08,9999-12-31,D19


=== ATTENDANCE DATE VALUES ===


,emp_id,work_date_key,work_date
0,E10160,2026-08-11,2026-08-11
1,E10160,2026-08-12,2026-08-12
2,E10160,2026-08-13,2026-08-13
3,E10160,2026-08-14,2026-08-14
4,E10160,2026-08-15,2026-08-15
5,E10160,2026-08-16,2026-08-16
6,E10160,2026-08-17,2026-08-17
7,E10160,2026-08-18,2026-08-18
8,E10160,2026-08-19,2026-08-19
9,E10160,2026-08-20,2026-08-20


In [16]:
from pathlib import Path
import re
import subprocess
import sys

project = Path.home() / "Documents" / "indus-weave-hr-pipeline"
script = project / "warehouse" / "stage4_warehouse_analytics.py"

source = script.read_text(encoding="utf-8")

pattern = r"def iso_date\(value\):.*?(?=\ndef num\()"

replacement = '''def iso_date(value):
    text = s(value)
    if not text:
        return None

    # Preserve ISO dates without swapping month and day.
    if len(text) >= 10 and text[4] == "-" and text[7] == "-":
        try:
            return datetime.strptime(
                text[:10], "%Y-%m-%d"
            ).date().isoformat()
        except ValueError:
            pass

    # Fallback for non-ISO source values.
    parsed = pd.to_datetime(
        text, errors="coerce", format="mixed", dayfirst=True
    )
    if pd.isna(parsed):
        return None

    return parsed.strftime("%Y-%m-%d")
'''

fixed, count = re.subn(
    pattern, replacement, source, count=1, flags=re.DOTALL
)

if count != 1:
    raise RuntimeError("Date parser patch apply nahi hua.")

compile(fixed, str(script), "exec")
script.write_text(fixed, encoding="utf-8")

print("Date parser fixed. Rerunning Stage 4-6...")

result = subprocess.run(
    [sys.executable, str(script)],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print("\nERROR:\n", result.stderr[-8000:])

print("\nExit code:", result.returncode)

Date parser fixed. Rerunning Stage 4-6...

===== RUNNING 10 ANALYTICS QUERIES =====
q01_headcount.sql: 150 rows -> q01_headcount.csv
q02_attrition.sql: 150 rows -> q02_attrition.csv
q03_absenteeism.sql: 24 rows -> q03_absenteeism.csv
q04_late_arrivals.sql: 10 rows -> q04_late_arrivals.csv
q05_overtime_reconciliation.sql: 150 rows -> q05_overtime_reconciliation.csv
q06_ot_overpayment.sql: 23 rows -> q06_ot_overpayment.csv
q07_paid_zero_attendance.sql: 59 rows -> q07_paid_zero_attendance.csv
q08_pay_after_separation.sql: 21 rows -> q08_pay_after_separation.csv
q09_payroll_cost.sql: 150 rows -> q09_payroll_cost.csv
q10_leave_overuse.sql: 20 rows -> q10_leave_overuse.csv
PDF generation unavailable: No module named 'reportlab'
Slide text saved to: /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/presentation/slides_content.md

STAGE 4-6 SUMMARY
dim_employee: 3,685 rows
dim_department: 25 rows
dim_shift: 4 rows
dim_date: 365 rows
fact_attendance_daily: 217,584 rows
fact_leave_day: 8,60

In [17]:
from pathlib import Path
import sqlite3
import subprocess
import sys
import importlib.util
import pandas as pd

project = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

db_path = project / "data" / "peopleops.sqlite"
results_dir = project / "analytics" / "results"
dq_path = project / "quality" / "dq_results.csv"
script_path = project / "warehouse" / "stage4_warehouse_analytics.py"

# 1. Data-quality summary
dq = pd.read_csv(dq_path)

print("===== DATA QUALITY SUMMARY =====")
print(dq["status"].value_counts(dropna=False).to_string())

problems = dq[dq["status"].astype(str).str.upper() != "PASS"]

print("\nChecks requiring review:", len(problems))

if not problems.empty:
    display(problems)

# 2. Verify warehouse table counts
print("\n===== WAREHOUSE TABLES =====")

tables = [
    "dim_employee",
    "dim_department",
    "dim_shift",
    "dim_date",
    "fact_attendance_daily",
    "fact_leave_day",
    "fact_overtime_approved",
    "fact_payroll_line",
]

with sqlite3.connect(db_path) as conn:
    for table in tables:
        count = conn.execute(
            f'SELECT COUNT(*) FROM "{table}"'
        ).fetchone()[0]
        print(f"{table}: {count:,}")

# 3. Confirm all 10 result CSVs exist
query_files = sorted(results_dir.glob("q*.csv"))

print("\nSQL result CSV files:", len(query_files))

for file in query_files:
    print(file.name)

# 4. Install ReportLab only if required
try:
    import reportlab
    print("\nReportLab is available.")
except ImportError:
    print("\nInstalling ReportLab...")
    install = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "reportlab",
        ],
        capture_output=True,
        text=True,
    )

    if install.returncode != 0:
        print("Installation failed:")
        print(install.stderr[-2500:])
    else:
        print("ReportLab installed successfully.")

# 5. Generate slides PDF from saved results without rebuilding
# the warehouse or rerunning all 10 SQL queries.
try:
    import reportlab

    spec = importlib.util.spec_from_file_location(
        "peopleops_stage4",
        script_path,
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    query_results = {
        file.stem: pd.read_csv(file)
        for file in query_files
    }

    pdf_path = module.create_presentation_pdf(
        query_results,
        dq,
    )

    if pdf_path and Path(pdf_path).exists():
        print("\nPRESENTATION PDF CREATED:", pdf_path)
    else:
        print("\nPDF was not created. Review the error above.")

except Exception as exc:
    print("\nPresentation error:", repr(exc))

===== DATA QUALITY SUMMARY =====
status
PASS    35
WARN     1

Checks requiring review: 1


,check_name,table_name,status,expected,actual,details,checked_at
35,Device clock correction documented,stg_clock_correction_evidence,WARN,Evidence row present,Present,Current evidence shows modest improvement only...,2026-10-25T02:16:18



===== WAREHOUSE TABLES =====
dim_employee: 3,685
dim_department: 25
dim_shift: 4
dim_date: 365
fact_attendance_daily: 217,584
fact_leave_day: 8,604
fact_overtime_approved: 19,666
fact_payroll_line: 53,617

SQL result CSV files: 10
q01_headcount.csv
q02_attrition.csv
q03_absenteeism.csv
q04_late_arrivals.csv
q05_overtime_reconciliation.csv
q06_ot_overpayment.csv
q07_paid_zero_attendance.csv
q08_pay_after_separation.csv
q09_payroll_cost.csv
q10_leave_overuse.csv

Installing ReportLab...
ReportLab installed successfully.

PRESENTATION PDF CREATED: /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/presentation/slides.pdf


In [18]:
from pathlib import Path
import pandas as pd

project = (
    Path.home()
    / "Documents"
    / "indus-weave-hr-pipeline"
)

# 1. Data quality results
dq_file = project / "quality" / "dq_results.csv"

print("===== DATA QUALITY =====")

if dq_file.exists():
    dq = pd.read_csv(dq_file)
    print(dq["status"].value_counts(dropna=False))
    print("\nChecks requiring attention:")
    display(
        dq[
            dq["status"].astype(str).str.upper() != "PASS"
        ]
    )
else:
    print("Data quality report missing!")

# 2. Verify required deliverables
required_files = [
    "README.md",
    "requirements.txt",
    "warehouse/erd.png",
    "presentation/cfo_findings.md",
    "presentation/slides.pdf",
]

print("\n===== DELIVERABLE CHECK =====")

for item in required_files:
    path = project / item
    print(f"{item}: {'EXISTS' if path.exists() else 'MISSING'}")

print(
    "\nSQL files:",
    len(list((project / "analytics").glob("q*.sql")))
)

print(
    "Result CSV files:",
    len(list((project / "analytics" / "results").glob("q*.csv")))
)

# 3. Display actual CFO findings
findings_file = project / "presentation" / "cfo_findings.md"

print("\n===== CFO FINDINGS =====")

if findings_file.exists():
    print(findings_file.read_text(encoding="utf-8"))
else:
    print("CFO findings file missing!")

===== DATA QUALITY =====
status
PASS    35
WARN     1
Name: count, dtype: int64

Checks requiring attention:


,check_name,table_name,status,expected,actual,details,checked_at
35,Device clock correction documented,stg_clock_correction_evidence,WARN,Evidence row present,Present,Current evidence shows modest improvement only...,2026-10-25T02:16:18



===== DELIVERABLE CHECK =====
README.md: EXISTS
requirements.txt: EXISTS
warehouse/erd.png: MISSING
presentation/cfo_findings.md: EXISTS
presentation/slides.pdf: EXISTS

SQL files: 10
Result CSV files: 10

===== CFO FINDINGS =====
# CFO Findings - Indus Weave Textiles

All numbers below are generated from the current local pipeline run. Recheck them after any transformation change.

## Finding 1 - Paid overtime vs punch-computed overtime
The largest flagged department/month is Stitching (Faisalabad Plant) in 2026-09: paid OT 1778.50 hours versus computed OT 729.50 hours, an excess of 1049.00 hours.
Evidence query: analytics/q06_ot_overpayment.sql. Recommended action: audit approval records, punch correction, roster assignment and payroll processing for this department/month.

## Finding 2 - Payroll paid with zero Present days
The query identified 59 employee-month rows with positive earnings and zero Present days under the documented definition.
Evidence query: analytics/q07_paid_zero

In [19]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

project = Path.home() / "Documents" / "indus-weave-hr-pipeline"
db_path = project / "data" / "peopleops.sqlite"
erd_path = project / "warehouse" / "erd.png"

# --------------------------------------
# 1. Create a warehouse ER diagram
# --------------------------------------
fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 15)
ax.set_ylim(0, 10)
ax.axis("off")

dimensions = [
    ("dim_employee", 0.5, 7.0),
    ("dim_department", 4.1, 7.0),
    ("dim_shift", 7.7, 7.0),
    ("dim_date", 11.3, 7.0),
]

facts = [
    ("fact_attendance_daily", 0.5, 3.0),
    ("fact_leave_day", 4.1, 3.0),
    ("fact_overtime_approved", 7.7, 3.0),
    ("fact_payroll_line", 11.3, 3.0),
]

box_w, box_h = 3.1, 1.35

def draw_box(name, x, y, kind):
    face = "#DCEAF7" if kind == "dimension" else "#E0F1E5"
    box = FancyBboxPatch(
        (x, y), box_w, box_h,
        boxstyle="round,pad=0.08",
        linewidth=1.5,
        edgecolor="#34495E",
        facecolor=face,
        zorder=3,
    )
    ax.add_patch(box)
    ax.text(
        x + box_w / 2, y + box_h / 2 + 0.12,
        name, ha="center", va="center",
        fontsize=10, fontweight="bold",
        zorder=4,
    )
    ax.text(
        x + box_w / 2, y + 0.27,
        kind.upper(), ha="center", va="center",
        fontsize=8, color="#34495E",
        zorder=4,
    )

for name, x, y in dimensions:
    draw_box(name, x, y, "dimension")

for name, x, y in facts:
    draw_box(name, x, y, "fact")

# Conceptual relationships: dimensions filter the relevant facts.
# Exact FK columns should be confirmed from the database DDL.
relations = [
    (0, 0), (0, 1), (0, 2), (0, 3),
    (1, 0), (1, 1), (1, 2), (1, 3),
    (2, 0), (2, 2),
    (3, 0), (3, 1), (3, 2), (3, 3),
]

for dim_index, fact_index in relations:
    _, dx, dy = dimensions[dim_index]
    _, fx, fy = facts[fact_index]

    start_x = dx + box_w / 2
    end_x = fx + box_w / 2

    arrow = FancyArrowPatch(
        (start_x, dy),
        (end_x, fy + box_h),
        arrowstyle="-|>",
        mutation_scale=10,
        linewidth=0.9,
        color="#7F8C8D",
        alpha=0.55,
        connectionstyle="arc3,rad=0.04",
        zorder=1,
    )
    ax.add_patch(arrow)

ax.text(
    7.5, 9.45,
    "INDUS WEAVE TEXTILES — WAREHOUSE STAR MODEL",
    ha="center", va="center", fontsize=15, fontweight="bold",
)
ax.text(
    7.5, 1.6,
    "Conceptual relationships; verify column-level foreign keys against the DDL.",
    ha="center", fontsize=9, color="#555555",
)

fig.savefig(erd_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print("ER diagram saved:", erd_path)
print("ER diagram exists:", erd_path.exists())

# --------------------------------------
# 2. Check data-quality results
# --------------------------------------
dq_path = project / "quality" / "dq_results.csv"

print("\n===== DATA QUALITY SUMMARY =====")

if dq_path.exists():
    dq = pd.read_csv(dq_path)
    print(dq["status"].value_counts(dropna=False).to_string())

    review = dq[
        dq["status"].astype(str).str.upper() != "PASS"
    ]

    print("\nChecks needing review:", len(review))

    if not review.empty:
        display(review)
else:
    print("Missing report:", dq_path)

# --------------------------------------
# 3. Print all CFO findings
# --------------------------------------
findings_path = project / "presentation" / "cfo_findings.md"

print("\n===== COMPLETE CFO FINDINGS =====")

if findings_path.exists():
    print(findings_path.read_text(encoding="utf-8"))
else:
    print("CFO findings file missing.")

# --------------------------------------
# 4. Verify slides and SQL deliverables
# --------------------------------------
print("\n===== FINAL FILE CHECK =====")

required = [
    project / "README.md",
    project / "requirements.txt",
    erd_path,
    project / "presentation" / "cfo_findings.md",
    project / "presentation" / "slides.pdf",
]

for path in required:
    print(
        f"{path.relative_to(project)}: "
        f"{'OK' if path.exists() else 'MISSING'}"
    )

print("SQL files:", len(list((project / "analytics").glob("q*.sql"))))
print(
    "Result CSVs:",
    len(list((project / "analytics" / "results").glob("q*.csv"))),
)

ModuleNotFoundError: No module named 'matplotlib'

In [22]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

project = Path.home() / "Documents" / "indus-weave-hr-pipeline"
db_path = project / "data" / "peopleops.sqlite"
erd_path = project / "warehouse" / "erd.png"

# --------------------------------------
# 1. Create a warehouse ER diagram
# --------------------------------------
fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 15)
ax.set_ylim(0, 10)
ax.axis("off")

dimensions = [
    ("dim_employee", 0.5, 7.0),
    ("dim_department", 4.1, 7.0),
    ("dim_shift", 7.7, 7.0),
    ("dim_date", 11.3, 7.0),
]

facts = [
    ("fact_attendance_daily", 0.5, 3.0),
    ("fact_leave_day", 4.1, 3.0),
    ("fact_overtime_approved", 7.7, 3.0),
    ("fact_payroll_line", 11.3, 3.0),
]

box_w, box_h = 3.1, 1.35

def draw_box(name, x, y, kind):
    face = "#DCEAF7" if kind == "dimension" else "#E0F1E5"
    box = FancyBboxPatch(
        (x, y), box_w, box_h,
        boxstyle="round,pad=0.08",
        linewidth=1.5,
        edgecolor="#34495E",
        facecolor=face,
        zorder=3,
    )
    ax.add_patch(box)
    ax.text(
        x + box_w / 2, y + box_h / 2 + 0.12,
        name, ha="center", va="center",
        fontsize=10, fontweight="bold",
        zorder=4,
    )
    ax.text(
        x + box_w / 2, y + 0.27,
        kind.upper(), ha="center", va="center",
        fontsize=8, color="#34495E",
        zorder=4,
    )

for name, x, y in dimensions:
    draw_box(name, x, y, "dimension")

for name, x, y in facts:
    draw_box(name, x, y, "fact")

# Conceptual relationships: dimensions filter the relevant facts.
# Exact FK columns should be confirmed from the database DDL.
relations = [
    (0, 0), (0, 1), (0, 2), (0, 3),
    (1, 0), (1, 1), (1, 2), (1, 3),
    (2, 0), (2, 2),
    (3, 0), (3, 1), (3, 2), (3, 3),
]

for dim_index, fact_index in relations:
    _, dx, dy = dimensions[dim_index]
    _, fx, fy = facts[fact_index]

    start_x = dx + box_w / 2
    end_x = fx + box_w / 2

    arrow = FancyArrowPatch(
        (start_x, dy),
        (end_x, fy + box_h),
        arrowstyle="-|>",
        mutation_scale=10,
        linewidth=0.9,
        color="#7F8C8D",
        alpha=0.55,
        connectionstyle="arc3,rad=0.04",
        zorder=1,
    )
    ax.add_patch(arrow)

ax.text(
    7.5, 9.45,
    "INDUS WEAVE TEXTILES — WAREHOUSE STAR MODEL",
    ha="center", va="center", fontsize=15, fontweight="bold",
)
ax.text(
    7.5, 1.6,
    "Conceptual relationships; verify column-level foreign keys against the DDL.",
    ha="center", fontsize=9, color="#555555",
)

fig.savefig(erd_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print("ER diagram saved:", erd_path)
print("ER diagram exists:", erd_path.exists())

# --------------------------------------
# 2. Check data-quality results
# --------------------------------------
dq_path = project / "quality" / "dq_results.csv"

print("\n===== DATA QUALITY SUMMARY =====")

if dq_path.exists():
    dq = pd.read_csv(dq_path)
    print(dq["status"].value_counts(dropna=False).to_string())

    review = dq[
        dq["status"].astype(str).str.upper() != "PASS"
    ]

    print("\nChecks needing review:", len(review))

    if not review.empty:
        display(review)
else:
    print("Missing report:", dq_path)

# --------------------------------------
# 3. Print all CFO findings
# --------------------------------------
findings_path = project / "presentation" / "cfo_findings.md"

print("\n===== COMPLETE CFO FINDINGS =====")

if findings_path.exists():
    print(findings_path.read_text(encoding="utf-8"))
else:
    print("CFO findings file missing.")

# --------------------------------------
# 4. Verify slides and SQL deliverables
# --------------------------------------
print("\n===== FINAL FILE CHECK =====")

required = [
    project / "README.md",
    project / "requirements.txt",
    erd_path,
    project / "presentation" / "cfo_findings.md",
    project / "presentation" / "slides.pdf",
]

for path in required:
    print(
        f"{path.relative_to(project)}: "
        f"{'OK' if path.exists() else 'MISSING'}"
    )

print("SQL files:", len(list((project / "analytics").glob("q*.sql"))))
print(
    "Result CSVs:",
    len(list((project / "analytics" / "results").glob("q*.csv"))),
)

Matplotlib is building the font cache; this may take a moment.


ER diagram saved: /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/warehouse/erd.png
ER diagram exists: True

===== DATA QUALITY SUMMARY =====
status
PASS    35
WARN     1

Checks needing review: 1


,check_name,table_name,status,expected,actual,details,checked_at
35,Device clock correction documented,stg_clock_correction_evidence,WARN,Evidence row present,Present,Current evidence shows modest improvement only...,2026-10-25T02:16:18



===== COMPLETE CFO FINDINGS =====
# CFO Findings - Indus Weave Textiles

All numbers below are generated from the current local pipeline run. Recheck them after any transformation change.

## Finding 1 - Paid overtime vs punch-computed overtime
The largest flagged department/month is Stitching (Faisalabad Plant) in 2026-09: paid OT 1778.50 hours versus computed OT 729.50 hours, an excess of 1049.00 hours.
Evidence query: analytics/q06_ot_overpayment.sql. Recommended action: audit approval records, punch correction, roster assignment and payroll processing for this department/month.

## Finding 2 - Payroll paid with zero Present days
The query identified 59 employee-month rows with positive earnings and zero Present days under the documented definition.
Evidence query: analytics/q07_paid_zero_attendance.sql. Recommended action: reconcile missing-punch exceptions, approved leave, roster/shift allocation and payroll eligibility before treating these as errors.

## Finding 3 - Payroll aft

In [21]:
import sys
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "matplotlib"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Matplotlib successfully installed!")
else:
    print("Installation error:")
    print(result.stderr[-3000:])

Matplotlib successfully installed!


In [23]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

project = Path.home() / "Documents" / "indus-weave-hr-pipeline"
db = project / "data" / "peopleops.sqlite"
erd = project / "warehouse" / "erd.png"

dims = ["dim_employee", "dim_department", "dim_shift", "dim_date"]
facts = [
    "fact_attendance_daily",
    "fact_leave_day",
    "fact_overtime_approved",
    "fact_payroll_line",
]
tables = dims + facts

# Read actual foreign keys from SQLite.
with sqlite3.connect(db) as con:
    relationships = []
    for child in tables:
        for fk in con.execute(
            f'PRAGMA foreign_key_list("{child}")'
        ).fetchall():
            parent = fk[2]
            child_col = fk[3]
            parent_col = fk[4]
            if parent in tables:
                relationships.append(
                    (parent, parent_col, child, child_col)
                )

# Draw dimensions on top and facts at bottom.
positions = {
    "dim_employee": (0.4, 6.4),
    "dim_department": (4.0, 6.4),
    "dim_shift": (7.6, 6.4),
    "dim_date": (11.2, 6.4),
    "fact_attendance_daily": (0.4, 2.8),
    "fact_leave_day": (4.0, 2.8),
    "fact_overtime_approved": (7.6, 2.8),
    "fact_payroll_line": (11.2, 2.8),
}

fig, ax = plt.subplots(figsize=(17, 10))
ax.set_xlim(0, 15.5)
ax.set_ylim(1.8, 8.7)
ax.axis("off")

box_w, box_h = 3.0, 1.15

for table, (x, y) in positions.items():
    color = "#DCEAF7" if table in dims else "#E0F1E5"

    ax.add_patch(FancyBboxPatch(
        (x, y), box_w, box_h,
        boxstyle="round,pad=0.08",
        facecolor=color, edgecolor="#34495E",
        linewidth=1.4, zorder=3
    ))

    ax.text(
        x + box_w / 2, y + box_h / 2,
        table, ha="center", va="center",
        fontsize=9, fontweight="bold", zorder=4
    )

# Connect only relationships present in actual SQLite DDL.
for parent, parent_col, child, child_col in relationships:
    px, py = positions[parent]
    cx, cy = positions[child]

    start = (px + box_w / 2, py)
    end = (cx + box_w / 2, cy + box_h)

    if parent == child:
        continue

    ax.add_patch(FancyArrowPatch(
        start, end,
        arrowstyle="-|>", mutation_scale=10,
        linewidth=1.0, color="#667788",
        alpha=0.7, connectionstyle="arc3,rad=0.05",
        zorder=1
    ))

ax.text(
    7.5, 8.15, "INDUS WEAVE TEXTILES — WAREHOUSE ERD",
    ha="center", fontsize=15, fontweight="bold"
)
ax.text(
    7.5, 2.15,
    f"Relationships found in database DDL: {len(relationships)}",
    ha="center", fontsize=9
)

fig.savefig(erd, dpi=180, bbox_inches="tight")
plt.close(fig)

print("ERD created:", erd.exists(), erd)

# Final verification
dq_file = project / "quality" / "dq_results.csv"
print("\n===== DATA QUALITY =====")

if dq_file.exists():
    dq = pd.read_csv(dq_file)
    print(dq["status"].value_counts(dropna=False).to_string())
    issues = dq[dq["status"].astype(str).str.upper() != "PASS"]
    print("\nChecks needing review:", len(issues))
    if not issues.empty:
        display(issues)
else:
    print("Data quality report missing.")

findings = project / "presentation" / "cfo_findings.md"
print("\n===== CFO FINDINGS =====")
print(
    findings.read_text(encoding="utf-8")
    if findings.exists()
    else "CFO findings file missing."
)

print("\n===== FINAL FILES =====")
for relative in [
    "README.md",
    "requirements.txt",
    "warehouse/erd.png",
    "presentation/cfo_findings.md",
    "presentation/slides.pdf",
]:
    path = project / relative
    print(f"{relative}: {'OK' if path.exists() else 'MISSING'}")

print("SQL files:", len(list((project / "analytics").glob("q*.sql"))))
print("Result CSVs:", len(list((project / "analytics/results").glob("q*.csv"))))

ERD created: True /Users/abdullahahmed/Documents/indus-weave-hr-pipeline/warehouse/erd.png

===== DATA QUALITY =====
status
PASS    35
WARN     1

Checks needing review: 1


,check_name,table_name,status,expected,actual,details,checked_at
35,Device clock correction documented,stg_clock_correction_evidence,WARN,Evidence row present,Present,Current evidence shows modest improvement only...,2026-10-25T02:16:18



===== CFO FINDINGS =====
# CFO Findings - Indus Weave Textiles

All numbers below are generated from the current local pipeline run. Recheck them after any transformation change.

## Finding 1 - Paid overtime vs punch-computed overtime
The largest flagged department/month is Stitching (Faisalabad Plant) in 2026-09: paid OT 1778.50 hours versus computed OT 729.50 hours, an excess of 1049.00 hours.
Evidence query: analytics/q06_ot_overpayment.sql. Recommended action: audit approval records, punch correction, roster assignment and payroll processing for this department/month.

## Finding 2 - Payroll paid with zero Present days
The query identified 59 employee-month rows with positive earnings and zero Present days under the documented definition.
Evidence query: analytics/q07_paid_zero_attendance.sql. Recommended action: reconcile missing-punch exceptions, approved leave, roster/shift allocation and payroll eligibility before treating these as errors.

## Finding 3 - Payroll after separa

In [24]:
from pathlib import Path
import pandas as pd

project = Path.home() / "Documents" / "indus-weave-hr-pipeline"

print("ERD exists:",
      (project / "warehouse" / "erd.png").exists())

dq_file = project / "quality" / "dq_results.csv"

if dq_file.exists():
    dq = pd.read_csv(dq_file)
    print("\nDATA QUALITY:")
    print(dq["status"].value_counts(dropna=False).to_string())
else:
    print("DQ report missing")

findings = project / "presentation" / "cfo_findings.md"

print("\nCFO FINDINGS:")
if findings.exists():
    print(findings.read_text(encoding="utf-8"))

ERD exists: True

DATA QUALITY:
status
PASS    35
WARN     1

CFO FINDINGS:
# CFO Findings - Indus Weave Textiles

All numbers below are generated from the current local pipeline run. Recheck them after any transformation change.

## Finding 1 - Paid overtime vs punch-computed overtime
The largest flagged department/month is Stitching (Faisalabad Plant) in 2026-09: paid OT 1778.50 hours versus computed OT 729.50 hours, an excess of 1049.00 hours.
Evidence query: analytics/q06_ot_overpayment.sql. Recommended action: audit approval records, punch correction, roster assignment and payroll processing for this department/month.

## Finding 2 - Payroll paid with zero Present days
The query identified 59 employee-month rows with positive earnings and zero Present days under the documented definition.
Evidence query: analytics/q07_paid_zero_attendance.sql. Recommended action: reconcile missing-punch exceptions, approved leave, roster/shift allocation and payroll eligibility before treating the

In [25]:
from pathlib import Path
import pandas as pd

project = Path.home() / "Documents" / "indus-weave-hr-pipeline"

# Warning ki exact detail
dq = pd.read_csv(project / "quality" / "dq_results.csv")

print("===== WARNING DETAILS =====")
display(dq[dq["status"].astype(str).str.upper() == "WARN"])

# Complete CFO findings
print("\n===== CFO FINDINGS =====")
findings = project / "presentation" / "cfo_findings.md"
print(findings.read_text(encoding="utf-8"))

# Final required files
print("\n===== FINAL CHECK =====")
for name in [
    "README.md",
    "requirements.txt",
    "warehouse/erd.png",
    "presentation/cfo_findings.md",
    "presentation/slides.pdf",
]:
    print(name, "OK" if (project / name).exists() else "MISSING")

===== WARNING DETAILS =====


,check_name,table_name,status,expected,actual,details,checked_at
35,Device clock correction documented,stg_clock_correction_evidence,WARN,Evidence row present,Present,Current evidence shows modest improvement only...,2026-10-25T02:16:18



===== CFO FINDINGS =====
# CFO Findings - Indus Weave Textiles

All numbers below are generated from the current local pipeline run. Recheck them after any transformation change.

## Finding 1 - Paid overtime vs punch-computed overtime
The largest flagged department/month is Stitching (Faisalabad Plant) in 2026-09: paid OT 1778.50 hours versus computed OT 729.50 hours, an excess of 1049.00 hours.
Evidence query: analytics/q06_ot_overpayment.sql. Recommended action: audit approval records, punch correction, roster assignment and payroll processing for this department/month.

## Finding 2 - Payroll paid with zero Present days
The query identified 59 employee-month rows with positive earnings and zero Present days under the documented definition.
Evidence query: analytics/q07_paid_zero_attendance.sql. Recommended action: reconcile missing-punch exceptions, approved leave, roster/shift allocation and payroll eligibility before treating these as errors.

## Finding 3 - Payroll after separa

In [1]:
from pathlib import Path
import pandas as pd

project = Path.home() / "Documents" / "indus-weave-hr-pipeline"
dq_file = project / "quality" / "dq_results.csv"

dq = pd.read_csv(dq_file)

display(
    dq[dq["status"].astype(str).str.upper() == "WARN"]
)

,check_name,table_name,status,expected,actual,details,checked_at
35,Device clock correction documented,stg_clock_correction_evidence,WARN,Evidence row present,Present,Current evidence shows modest improvement only...,2026-10-25T02:16:18
